# Bonus Lab 7: Artificial Neural Networks (ANN)

*FW 532 · Python version*

*This is an optional (bonus) lab that goes with the neural networks lecture.*

**Goals:**

* To understand how neural network models are built and interpreted in Python.

**You Will Need:**

* The course environmental raster stack (`data/envStack.tif`, included in the starter kit)
* Python packages: numpy, pandas, matplotlib, scikit-learn, rasterio (see `requirements.txt`)
* The course helper module `sdmlab.py` (in the same folder as this notebook)

Remember to run the cells in order from top to bottom.

In [ ]:
# Running on Google Colab? This cell downloads the course files. (Anywhere else it does nothing.)
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/fw532"):
        !git clone -q --depth 1 https://github.com/Hutchinson-Lab/fw532.git /content/fw532
        %pip install -q rasterio
    os.chdir("/content/fw532/python")

# Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier   # neural networks
from sklearn.metrics import roc_auc_score          # AUC

import sdmlab as sdm   # course helpers for rasters

env_stack = sdm.load_env()

# Generate Data

We simulate the same virtual species as the R version: a quadratic effect of Mean.Diurnal.Range
and an interaction between Isothermality and Mean.Temperature.of.Wettest.Quarter. (The random
points differ from R's, because Python and R use different random number generators, so your
numbers won't match the R lab exactly.)

In [ ]:
# define logistic function
def logistic(x):
    return np.exp(x) / (1 + np.exp(x))

# here's the scaling: standardize every layer once (mean 0, sd 1 over all cells)
scaled_stack = env_stack.scale()

# get points (from the scaled stack, so the data are already scaled)
n_points = 1000
sim_pts = scaled_stack.sample_random(n_points, seed=1234)

# set parameters
intercept = -0.25
beta1 = -1
beta2 = 1
beta3 = 1.5
beta4 = 1
beta5 = -1
p_occ = logistic(intercept
                 + beta1 * scaled_stack["Mean.Diurnal.Range"]
                 + beta2 * scaled_stack["Isothermality"]
                 + beta3 * scaled_stack["Mean.Temperature.of.Wettest.Quarter"]
                 + beta4 * scaled_stack["Mean.Diurnal.Range"]**2   # quadratic term
                 + beta5 * scaled_stack["Isothermality"] * scaled_stack["Mean.Temperature.of.Wettest.Quarter"])  # interaction term

# get the values that correspond to the points we selected:
p_occs = sdm.extract(p_occ, env_stack, sim_pts["x"], sim_pts["y"])

# sample from these probabilities (with a different seed -- see Lab 1):
rng = np.random.default_rng(5678)
pres_abs = rng.binomial(n=1, p=p_occs)

# put response and variables together in a DataFrame:
modeling_df = sim_pts.copy()
modeling_df.insert(0, "presAbs", pres_abs)

# plot the result
ax = sdm.plot_map(p_occ, env_stack, title="True Prob. of Occurrence", vmin=0, vmax=1)
absent = modeling_df["presAbs"] == 0
ax.scatter(modeling_df.loc[absent, "x"], modeling_df.loc[absent, "y"], s=4, color="blue")
ax.scatter(modeling_df.loc[~absent, "x"], modeling_df.loc[~absent, "y"], s=4, color="red");

In [ ]:
# split into training and test
# (sample_random() returns the points in random order, so this is a random split)
train_data = modeling_df.iloc[:n_points // 2]
test_data = modeling_df.iloc[n_points // 2:]

> **Why scale the whole stack once?** Neural networks are sensitive to the scale of their
> inputs, so we standardize every variable. It matters that the *same* standardization is used
> everywhere. A tempting shortcut is to standardize the 1,000 sampled points with their own means
> and SDs, and later standardize the grid of all cells with the grid's means and SDs. The two sets
> of means and SDs differ slightly, so the map would be predicted from inputs on a slightly
> different scale than the ones the model was trained on. Scaling the raster stack once, then
> sampling the training data from it and predicting onto it, avoids that problem: a value of 1.2
> for a variable means the same thing in training and in prediction.

We'll make predictions for every cell and map them back to geographic space to visualize models.
As in earlier labs, `scaled_stack.to_frame()` gives a table with one row per land cell; we predict
for every row and put the predictions back on the map with `to_map()`. Because the table comes from
`scaled_stack`, the map inputs are scaled exactly like the training data.

Most of our models use all 20 variables, so we'll keep their names in a list:

In [ ]:
pred_vars = env_stack.names
grid = scaled_stack.to_frame()   # one row per land cell
print(len(pred_vars), "variables;", len(grid), "cells to predict")

# Neural Networks

## Fit with default parameters

Let's start with a basic model - all variables, mostly default parameters. In Python we use
scikit-learn's `MLPClassifier` (MLP = "multi-layer perceptron", another name for this kind of
network). We set a few options so that it fits the same kind of network as R's neuralnet:

* `hidden_layer_sizes=(6,)` -- the network architecture: one layer of six hidden units, chosen
  arbitrarily. You can also experiment with more layers, e.g. `(8, 4)` gives 8 hidden units in the
  first layer and 4 in the second. (The trailing comma in `(6,)` makes it a *tuple* with one
  element; `(6)` would just be the number 6.)
* `activation="logistic"` -- hidden units use the sigmoid/logistic function, like neuralnet
  (scikit-learn's default is `"relu"`). For a 0/1 response, `MLPClassifier` always puts the
  output through the logistic function, so predictions are probabilities (like
  `linear.output = FALSE` in R).
* `solver="lbfgs"` -- the fitting algorithm. L-BFGS works well for small datasets like ours.
  (neuralnet uses a different algorithm, resilient backpropagation.)
* `alpha=0` -- no weight decay. scikit-learn's default (`alpha=0.0001`) adds a small penalty on
  large weights, a form of regularization; neuralnet has no such penalty, so we turn it off.
* `max_iter=2000` -- allow plenty of iterations, so training stops because the stopping rule is
  met rather than because time ran out (like neuralnet's `stepmax`).
* `random_state=7` -- the weights are initialized to small random numbers at the beginning of the
  training process, so we fix the seed to get consistent results across runs. We use the same seed
  for every network, so networks with the same architecture start from the same weights.

One more difference: neuralnet (by default) fits the weights by minimizing the sum of squared
errors, while scikit-learn minimizes the cross-entropy (log-loss), which is the binomial deviance
that logistic regression minimizes. Both push predictions toward the observed 0s and 1s.

In [ ]:
model1 = MLPClassifier(hidden_layer_sizes=(6,), activation="logistic", solver="lbfgs",
                       alpha=0, max_iter=2000, random_state=7)
model1.fit(train_data[pred_vars], train_data["presAbs"])
print("iterations:", model1.n_iter_)

scikit-learn has no picture of the network like neuralnet's `plot()`, but the picture is just a
way to show the fitted **weights**, and we can look at those directly. `model1.coefs_[0]` holds the
input → hidden weights (a 20 × 6 table: one row per variable, one column per hidden unit),
`model1.intercepts_[0]` holds each hidden unit's bias (intercept), and `model1.coefs_[1]` and
`model1.intercepts_[1]` hold the hidden → output weights and the output bias. The function below
draws them as a heatmap with the numbers written in. We define it once and reuse it for every model.

In [ ]:
def plot_weights(model, var_names, title=""):
    """Heatmap of input->hidden weights (plus biases) and a bar chart of hidden->output weights."""
    w_in = np.vstack([model.intercepts_[0], model.coefs_[0]])   # first row = biases
    row_names = ["(bias)"] + list(var_names)
    w_out = model.coefs_[1][:, 0]
    n_hidden = w_in.shape[1]
    unit_names = [f"H{j + 1}" for j in range(n_hidden)]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 0.35 * len(row_names) + 1.5),
                                   gridspec_kw={"width_ratios": [2, 1]}, constrained_layout=True)
    limit = np.abs(w_in).max()
    im = ax1.imshow(w_in, cmap="RdBu_r", vmin=-limit, vmax=limit, aspect="auto")
    for i in range(w_in.shape[0]):
        for j in range(n_hidden):
            ax1.text(j, i, f"{w_in[i, j]:.1f}", ha="center", va="center", fontsize=7)
    ax1.set_xticks(range(n_hidden), unit_names)
    ax1.set_yticks(range(len(row_names)), row_names, fontsize=8)
    ax1.set_title("input → hidden weights")
    fig.colorbar(im, ax=ax1, shrink=0.6, label="weight")

    ax2.bar(unit_names, w_out, color=np.where(w_out > 0, "tab:red", "tab:blue"))
    ax2.axhline(0, color="black", linewidth=0.8)
    ax2.set_title(f"hidden → output weights\n(output bias = {model.intercepts_[1][0]:.1f})")
    fig.suptitle(title)
    return fig

plot_weights(model1, pred_vars, "model1: 6 hidden units, all variables");

**How to read this.** Each column of the heatmap is one hidden unit. A hidden unit computes
$h_j = \text{logistic}(\text{bias}_j + \sum_i w_{ij} x_i)$, so each number in the column says how
strongly (and in which direction: red = positive, blue = negative) that variable pushes the unit
toward 1 or 0. The bar chart shows how each hidden unit pushes the log-odds of presence up (red)
or down (blue): the output is $\text{logistic}(\text{bias} + \sum_j v_j h_j)$. Be careful about
reading meaning into single weights: the hidden units are interchangeable (shuffling the columns
gives the same model), and a unit can flip all its signs if its output weight flips too. Look for
overall patterns, especially *how big* the weights are.

How well does this model predict the test observations? Let's measure with AUC for now.
`predict_proba()` returns two columns (probability of 0 and of 1); `[:, 1]` keeps the probability
of presence.

In [ ]:
preds1train = model1.predict_proba(train_data[pred_vars])[:, 1]
print("training AUC:", round(roc_auc_score(train_data["presAbs"], preds1train), 4))

preds1test = model1.predict_proba(test_data[pred_vars])[:, 1]
print("test AUC:    ", round(roc_auc_score(test_data["presAbs"], preds1test), 4))

Training AUC is substantially better than test AUC.

## Visualize the predictive model

We will also want to look at our predictions in geographic space as we experiment with these
models. This small function predicts every cell and plots the map next to the truth.

In [ ]:
def plot_prediction(model, variables):
    """Predict every land cell and plot the map next to the true probability of occurrence."""
    pred = scaled_stack.to_map(model.predict_proba(grid[variables])[:, 1], index=grid.index)
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), constrained_layout=True)
    sdm.plot_map(p_occ, env_stack, ax=axes[0], title="True Prob. of Occurrence", vmin=0, vmax=1)
    sdm.plot_map(pred, env_stack, ax=axes[1], title="NN prediction", vmin=0, vmax=1)
    return pred

pred_map = plot_prediction(model1, pred_vars)

## Model Tuning

The basic model above looks overfit! Evidence of this: 1) some pretty large magnitude weights for
variables ~N(0,1) and hidden units using the sigmoid/logistic activation (i.e. outputs in [0,1]) -
imagine getting regression coefficients (i.e. betas) of 50 with standardized inputs; 2) big gap
between training set and testing set performance; 3) extreme predictions across the geographic
extent (i.e. lots of predictions near 0 (dark purple) and near 1 (yellow) in that map - see also a
histogram of the predictions, below).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(pred_map[~np.isnan(pred_map)], bins=50)
ax.set_xlabel("predicted probability (all cells)")
ax.set_ylabel("number of cells");

So, what to do about this? There are a few ways to influence the complexity of the neural net.
We'll look at the network architecture, the input variables, and the number of iterations the
fitting algorithm is run for.

### Simpler architecture

Let's halve the number of hidden units. This will take us from 20x6 parameters in the first layer
to 20x3.

In [ ]:
model2 = MLPClassifier(hidden_layer_sizes=(3,), activation="logistic", solver="lbfgs",
                       alpha=0, max_iter=2000, random_state=7)
model2.fit(train_data[pred_vars], train_data["presAbs"])
plot_weights(model2, pred_vars, "model2: 3 hidden units, all variables");

In [ ]:
preds2train = model2.predict_proba(train_data[pred_vars])[:, 1]
print("training AUC:", round(roc_auc_score(train_data["presAbs"], preds2train), 4))

preds2test = model2.predict_proba(test_data[pred_vars])[:, 1]
print("test AUC:    ", round(roc_auc_score(test_data["presAbs"], preds2test), 4))

In [ ]:
pred_map = plot_prediction(model2, pred_vars)

Some improvement! (In our run, the test AUC went up a little and the gap between training and test
AUC shrank. Your exact numbers may differ slightly with other package versions.)

### Fewer variables

Another option would be to reduce the set of input variables. Let's remove a bunch of the
precipitation variables, which are all correlated with each other anyway. Here, we'll go back to 6
hidden units again (changing one thing at a time), but we've still reduced the number of
parameters, now from 20x6 to 13x6. (We write the list out, so you can see which variables were
removed. The three variables marked "true" are the ones we used to simulate the species.)

In [ ]:
vars3 = ["Annual.Mean.Temperature",
         "Mean.Diurnal.Range",                   # true
         "Isothermality",                        # true
         "Temperature.Seasonality",
         "Max.Temperature.of.Warmest.Month",
         "Min.Temperature.of.Coldest.Month",
         "Temperature.Annual.Range",
         "Mean.Temperature.of.Wettest.Quarter",  # true
         # "Mean.Temperature.of.Driest.Quarter",
         "Mean.Temperature.of.Warmest.Quarter",
         # "Mean.Temperature.of.Coldest.Quarter",
         "Annual.Precipitation",
         # "Precipitation.of.Wettest.Month",
         # "Precipitation.of.Driest.Month",
         "Precipitation.Seasonality",
         "Precipitation.of.Wettest.Quarter",
         # "Precipitation.of.Driest.Quarter",
         # "Precipitation.of.Warmest.Quarter",
         # "Precipitation.of.Coldest.Quarter",
         "elevation"]

model3 = MLPClassifier(hidden_layer_sizes=(6,), activation="logistic", solver="lbfgs",
                       alpha=0, max_iter=2000, random_state=7)
model3.fit(train_data[vars3], train_data["presAbs"])
plot_weights(model3, vars3, "model3: 6 hidden units, 13 variables");

In [ ]:
preds3train = model3.predict_proba(train_data[vars3])[:, 1]
print("training AUC:", round(roc_auc_score(train_data["presAbs"], preds3train), 4))

preds3test = model3.predict_proba(test_data[vars3])[:, 1]
print("test AUC:    ", round(roc_auc_score(test_data["presAbs"], preds3test), 4))

In [ ]:
pred_map = plot_prediction(model3, vars3)

In this case, it looks like reducing the number of variables is not as effective as reducing the
number of hidden units in mitigating overfitting.

### Early stopping

Neural nets are trained with an iterative algorithm that continually updates the weight parameters
in the network. The longer it runs, the more it can adjust the parameters to fit the training set
better and better. The algorithm stops when some stopping criterion is reached.

In R, this is controlled by neuralnet's `threshold` parameter: the algorithm stops when the partial
derivatives (the gradient) of the error function all drop below the threshold (default 0.01), and
the R lab tries 0.1 instead. scikit-learn's L-BFGS solver has a very similar rule, controlled by
`tol`: it stops when every partial derivative of its loss is below `tol` (default `1e-4`). So
we'll make `tol` 10 times larger, `1e-3`, just as the R lab makes `threshold` 10 times larger.
This will cut the process off earlier and produce a simpler model, which we may hope will
generalize better.

The numbers themselves don't carry over between R and Python. neuralnet's error is a *sum* of
squared errors over the training points, while scikit-learn's loss is an *average* cross-entropy,
so their gradients are on very different scales. L-BFGS also stops if the loss stops decreasing,
or when it reaches `max_iter`. Also note that neither rule is "early stopping" in the sense you'll
often read about, where training stops when performance on a held-out *validation* set stops
improving (scikit-learn offers that with `early_stopping=True`, but only for its `"adam"` and
`"sgd"` solvers). Here we simply stop the optimizer sooner.

Because `model4` has the same architecture and seed as `model1`, it starts from the same weights
and follows the same path: it is `model1`'s training run, stopped earlier. Compare the number of
iterations.

In [ ]:
model4 = MLPClassifier(hidden_layer_sizes=(6,), activation="logistic", solver="lbfgs",
                       alpha=0, max_iter=2000, tol=1e-3, random_state=7)
model4.fit(train_data[pred_vars], train_data["presAbs"])
print("iterations: model1", model1.n_iter_, " model4", model4.n_iter_)
plot_weights(model4, pred_vars, "model4: 6 hidden units, all variables, tol = 1e-3");

In [ ]:
preds4train = model4.predict_proba(train_data[pred_vars])[:, 1]
print("training AUC:", round(roc_auc_score(train_data["presAbs"], preds4train), 4))

preds4test = model4.predict_proba(test_data[pred_vars])[:, 1]
print("test AUC:    ", round(roc_auc_score(test_data["presAbs"], preds4test), 4))

In [ ]:
pred_map = plot_prediction(model4, pred_vars)

# Model Interpretation

Supposing we've found a model whose fit we're satisfied with, we may want to understand more about
the relationships of inputs to outputs.

R's neuralnet package provides a function `gwplot()` to plot the 'generalized weights' for the
network. scikit-learn doesn't, but we can compute them ourselves. Intrator & Intrator (2001) (full
citation below) define the generalized weight of variable $x_i$ at a data point $x$ as the slope of
the predicted log-odds with respect to that variable, holding the other inputs fixed:

$$\tilde w_i(x) = \frac{\partial}{\partial x_i} \log\frac{\hat p(x)}{1 - \hat p(x)}.$$

We'll estimate this slope numerically, with a *finite difference*: nudge the variable up and down
by a tiny amount $h$ at each training point, and see how much the log-odds change:
$\tilde w_i(x) \approx \big[\text{logodds}(x_i + h) - \text{logodds}(x_i - h)\big] / 2h$.
(neuralnet computes the same derivative exactly, with calculus.)

First we need the log-odds. Rather than taking $\log(p / (1-p))$ of a probability that may be
rounded to exactly 0 or 1, we compute the network's output *before* the final logistic step,
using the weights we plotted above. This also shows that a fitted network is just a formula.

In [ ]:
def nn_log_odds(model, X):
    """Log-odds of presence from a one-hidden-layer MLPClassifier, computed from its weights."""
    hidden = logistic(np.asarray(X) @ model.coefs_[0] + model.intercepts_[0])
    return (hidden @ model.coefs_[1] + model.intercepts_[1])[:, 0]

# check: putting the log-odds through the logistic function gives predict_proba()'s answer
X_train = train_data[pred_vars]
print(np.allclose(logistic(nn_log_odds(model4, X_train)), model4.predict_proba(X_train)[:, 1]))

In [ ]:
def generalized_weights(model, X, variable, h=1e-4):
    """Finite-difference slope of the log-odds with respect to one variable, at each row of X."""
    X_up = X.copy()
    X_up[variable] = X_up[variable] + h
    X_down = X.copy()
    X_down[variable] = X_down[variable] - h
    return (nn_log_odds(model, X_up) - nn_log_odds(model, X_down)) / (2 * h)

gw = generalized_weights(model4, X_train, "Isothermality")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(X_train["Isothermality"], gw, facecolors="none", edgecolors="black")
ax.set_xlabel("Isothermality (scaled)")
ax.set_ylabel("GW")
ax.set_title("Response: presAbs");

There is a point for each data point in the training dataset. That paper states: "in neural
network modeling, the generalized weights have the same interpretation as weights have in logistic
regression, i.e., the contribution to the log odds. However, unlike logistic regression, this
contribution is local at each specific point x. The dependence of the generalized weights on the
specific covariate levels attests to the non-linearity of the ANN model, even with respect to the
log-odds. For example, it is possible that the same variable may have a positive effect for some
of the observations and a negative effect for others and its average effect may be close to zero.
The distribution of the generalized weights, over all the data shows whether a certain variable
has an overall strong effect, and determines if the effect is linear or not. A small variance of
the distribution suggests that the effect is linear. A large variance suggests that the effect is
non-linear as it varies over the observation space. In contrast, in logistic regression the
respective distribution is concentrated at one value." I don't have great intuition for this yet -
please share if you have insight on these plots!

> **What does the truth look like?** Because the inputs are standardized, a generalized weight is
> the change in log-odds per standard deviation of Isothermality. Because we simulated these data,
> we know the true slope too: the true log-odds contain $\beta_2 \cdot \text{Isothermality} +
> \beta_5 \cdot \text{Isothermality} \times \text{Mean.Temperature.of.Wettest.Quarter}$, so the true
> generalized weight for Isothermality is
> $\beta_2 + \beta_5 \cdot \text{Mean.Temperature.of.Wettest.Quarter}$, which changes from point
> to point.

## Permutation tests

One of the variable importance measures from random forests worked by permuting variables, thus
breaking the connection between the variable and the response, and measuring the effect on
performance. This is a general strategy that can be used for any model. Here's a start at how to
look at this. (scikit-learn also has a ready-made version, `sklearn.inspection.permutation_importance`,
but the loop below shows exactly what happens.)

In [ ]:
# set up a Series to record the training set AUC when each variable is permuted
permuted_aucs = pd.Series(0.0, index=pred_vars)

rng = np.random.default_rng(2468)
for v in pred_vars:
    # make a copy of the data
    perm_data = train_data.copy()
    # permute the variable's observed values
    perm_data[v] = rng.permutation(perm_data[v].values)
    # make predictions on the modified dataset
    preds4perm = model4.predict_proba(perm_data[pred_vars])[:, 1]
    # record the resulting AUC
    permuted_aucs[v] = roc_auc_score(train_data["presAbs"], preds4perm)

# without any permutation, we have a baseline for the AUC
preds4train = model4.predict_proba(train_data[pred_vars])[:, 1]
print("Base training AUC:", round(roc_auc_score(train_data["presAbs"], preds4train), 3))

permuted_aucs.round(3).sort_values()

For any given variable, you could turn this into a statistical test by permuting it many times and
repeating the process above. This would give you an empirical distribution of the null hypothesis
that the variable has no effect on the response. Then you could look at how different the
performance is when the variable IS included (the base training AUC above).

> **Exercise:** Play! Can you find a network structure that works better? Can you imagine using
> cross-validation to tune aspects of the network structure? Are you satisfied with the permutation
> importance approach, or do you want to try building a null distribution for a variable or two?
> (In Python you can also try weight decay: set `alpha` to a positive value such as 0.01 or 1.)

# Turn in

**(a Canvas discussion post, by the due date on Canvas)** Please post to the online discussion of
this lab. Please include brief descriptions of your exploration of the exercise, with any relevant
plots. Please also include, if applicable:

* Any key take-aways from this lab that you want to remember for any future projects you may
  conduct in which you use species distribution models.
* Any remaining questions or confusions do you have that you might like to learn more about.

# Further Resources

[Online book: *Neural Networks and Deep Learning*](http://neuralnetworksanddeeplearning.com/index.html)
(Michael Nielsen)

Lek, S., & Guégan, J. F. (1999). Artificial neural networks as a tool in ecological modelling, an
introduction. *Ecological Modelling*, 120, 65–73.

Generalized weights in ANNs:
Intrator, O., & Intrator, N. (2001). Interpreting neural-network results: a simulation study.
*Computational Statistics & Data Analysis*, 37, 373–393.